In [1]:
import os
import pandas as pd

# 1. Aseguramos que la carpeta exista
os.makedirs("data/raw", exist_ok=True)

# 2. Asignamos el identificador del IDEAM
dataset_id_ideam = "57sv-p2fu"
sample_rows = 150_000  # Límite ajustado para mayor velocidad

# 3. Construimos la URL
source_url_ideam = f"https://www.datos.gov.co/resource/{dataset_id_ideam}.csv?$limit={sample_rows}"

# 4. Descargamos y guardamos con mensajes de progreso
print("1. Conectando con datos.gov.co y descargando datos del IDEAM...")
df_ideam = pd.read_csv(source_url_ideam, low_memory=False)

print("2. Guardando archivo localmente en data/raw/ideam_sample.csv...")
df_ideam.to_csv("data/raw/ideam_sample.csv", index=False)

print("\n--- ¡Descarga e inspección finalizadas! ---")
print(f"Filas descargadas: {len(df_ideam):,}")
print(f"Columnas: {df_ideam.shape[1]}")

1. Conectando con datos.gov.co y descargando datos del IDEAM...
2. Guardando archivo localmente en data/raw/ideam_sample.csv...

--- ¡Descarga e inspección finalizadas! ---
Filas descargadas: 150,000
Columnas: 13


In [2]:
import os

def get_file_size_gb(file_path):
    """Tamaño real del archivo en disco, en gigabytes."""
    return os.path.getsize(file_path) / (1024 ** 3)

In [3]:
import pandas as pd

def measure_expansion_factor(file_path, **read_options):
    """Mide cuántas veces crece un archivo al cargarse en memoria."""
    size_disk_bytes = os.path.getsize(file_path)
    df = pd.read_csv(file_path, **read_options)
    size_memory_bytes = df.memory_usage(deep=True).sum()
    return size_memory_bytes / size_disk_bytes, df

In [4]:
import psutil

def get_available_memory_gb():
    """Memoria realmente disponible, descontando la que ya está en uso."""
    return psutil.virtual_memory().available / (1024 ** 3)

In [5]:
def profile_source(df, source_name):
    """Perfil mínimo de una fuente cargada."""
    dtype_counts = df.dtypes.value_counts().to_dict()
    object_columns = (df.dtypes == "object").sum()

    return {
        "fuente": source_name,
        "filas": len(df),
        "columnas": df.shape[1],
        "columnas_texto": int(object_columns),
        "proporcion_texto": round(object_columns / df.shape[1], 3),
        "tipos": {str(key): int(value) for key, value in dtype_counts.items()},
    }

In [8]:
import os
import pandas as pd
from pathlib import Path

# Funciona igual dentro del contenedor y fuera de el
BASE = Path("/home/jovyan/work")
if not BASE.exists():
    BASE = Path.cwd().parent          # notebooks/ -> raiz del repo
RAW = BASE / "data" / "raw"
OUT = BASE / "resultados"
OUT.mkdir(exist_ok=True)

sources = {
    "SECOP II": (RAW / "secop_sample.csv",
                 {"low_memory": False}),
    "IDEAM":    (RAW / "ideam_sample.csv",
                 {"low_memory": False}),
    # GEIH: separador ';' y codificacion latin-1. Sin el sep, pandas
    # lee UNA sola columna y el k sale falso (1.16 en vez de 5.37).
    "GEIH":     (RAW / "geih" / "Ocupados.CSV",
                 {"low_memory": False, "encoding": "latin-1", "sep": ";"}),
}

measurements = []
for source_name, (file_path, opts) in sources.items():
    k_value, df = measure_expansion_factor(str(file_path), **opts)
    profile = profile_source(df, source_name)
    profile["tamano_disco_gb"] = round(get_file_size_gb(str(file_path)), 6)
    profile["k"] = round(k_value, 2)
    measurements.append(profile)
    del df

df_measurements = pd.DataFrame(measurements)
df_measurements.to_csv(OUT / "mediciones.csv", index=False)
print(df_measurements[["fuente","filas","columnas","proporcion_texto","k"]])

     fuente   filas  columnas  proporcion_texto     k
0  SECOP II  200000        85             0.765  3.37
1     IDEAM  150000        13             0.615  3.24
2      GEIH   29611       202             0.054  5.37


In [9]:
def verify_level_1(df_measurements):
    """Verificación automática del nivel 1. Devuelve True si todo pasa."""
    checks = {
        "Hay exactamente 3 fuentes medidas": len(df_measurements) == 3,
        "Todos los k son mayores que 1": (df_measurements["k"] > 1).all(),
        "Los tres k son distintos": df_measurements["k"].nunique() == 3,
        "No hay tamaños de disco en cero": (df_measurements["tamano_disco_gb"] > 0).all(),
    }
    for description, passed in checks.items():
        print(f"{'PASA ' if passed else 'FALLA'} · {description}")
    return all(checks.values())

verify_level_1(df_measurements)

PASA  · Hay exactamente 3 fuentes medidas
PASA  · Todos los k son mayores que 1
PASA  · Los tres k son distintos
PASA  · No hay tamaños de disco en cero


True

In [11]:
import math

def compute_threshold_periods(memory_useful_gb, expansion_factor,
                              initial_size_gb, growth_rate):
    """Períodos que faltan para que la fuente exceda la memoria útil.

    Un resultado negativo indica que el umbral ya fue superado.
    """
    if growth_rate <= 0:
        raise ValueError("La tasa de crecimiento debe ser mayor que cero.")

    ratio = memory_useful_gb / (expansion_factor * initial_size_gb)
    return math.log(ratio) / math.log(1 + growth_rate)

In [12]:
g = 0.05   # Supuesto

M_real = get_available_memory_gb()
print(f"Memoria util medida en este equipo: {M_real:.2f} GB\n")

escenarios = {
    "8 GB nominales":  8 * 0.55,     # justifiquen la proporcion
    "16 GB nominales": 16 * 0.70,    # justifiquen la proporcion
    "Este equipo (medido)": M_real,
}

for etiqueta, M in escenarios.items():
    print(f"=== {etiqueta} -> M = {M:.2f} GB utiles ===")
    for fila in df_measurements.itertuples():
        t = compute_threshold_periods(M, fila.k, fila.tamano_disco_gb, g)
        estado = "YA SATURADO" if t < 0 else "margen"
        print(f"  {fila.fuente:10s} k={fila.k:5.2f}  S0={fila.tamano_disco_gb:.4f} GB"
              f"  ->  t = {t:7.1f} periodos  ({estado})")
    print()

Memoria util medida en este equipo: 6.11 GB

=== 8 GB nominales -> M = 4.40 GB utiles ===
  SECOP II   k= 3.37  S0=0.3045 GB  ->  t =    29.8 periodos  (margen)
  IDEAM      k= 3.24  S0=0.0321 GB  ->  t =    76.7 periodos  (margen)
  GEIH       k= 5.37  S0=0.0097 GB  ->  t =    91.0 periodos  (margen)

=== 16 GB nominales -> M = 11.20 GB utiles ===
  SECOP II   k= 3.37  S0=0.3045 GB  ->  t =    49.0 periodos  (margen)
  IDEAM      k= 3.24  S0=0.0321 GB  ->  t =    95.9 periodos  (margen)
  GEIH       k= 5.37  S0=0.0097 GB  ->  t =   110.2 periodos  (margen)

=== Este equipo (medido) -> M = 6.11 GB utiles ===
  SECOP II   k= 3.37  S0=0.3045 GB  ->  t =    36.6 periodos  (margen)
  IDEAM      k= 3.24  S0=0.0321 GB  ->  t =    83.5 periodos  (margen)
  GEIH       k= 5.37  S0=0.0097 GB  ->  t =    97.7 periodos  (margen)



In [13]:
FUENTE = "SECOP II"    # cambien por la fuente unica del equipo
f = df_measurements.set_index("fuente").loc[FUENTE]
M = get_available_memory_gb()

print(f"{FUENTE}: k={f.k}, S0={f.tamano_disco_gb} GB, M={M:.2f} GB\n")
for g_i in [0.01, 0.02, 0.04, 0.08, 0.16]:
    t = compute_threshold_periods(M, f.k, f.tamano_disco_gb, g_i)
    print(f"g = {g_i:.0%}  ->  t_umbral = {t:8.1f} periodos")

SECOP II: k=3.37, S0=0.304458 GB, M=6.11 GB

g = 1%  ->  t_umbral =    179.3 periodos
g = 2%  ->  t_umbral =     90.1 periodos
g = 4%  ->  t_umbral =     45.5 periodos
g = 8%  ->  t_umbral =     23.2 periodos
g = 16%  ->  t_umbral =     12.0 periodos


In [14]:
import pandas as pd
from pathlib import Path

# Usamos una ruta relativa que busca la carpeta 'data' un nivel arriba de 'notebooks'
ruta_archivo = Path("../data/raw/secop_sample.csv")

# 1. Cargamos la muestra cruda de SECOP II
df_secop = pd.read_csv(ruta_archivo, low_memory=False)

# 2. Convertimos la columna de fecha (la firma del contrato es el hito principal)
ts = pd.to_datetime(df_secop["fecha_de_firma"], errors='coerce')

# 3. Calculamos la diferencia de tiempo (delta) entre registros consecutivos
print("Frecuencia observada en la muestra de SECOP II:")
print(ts.sort_values().diff().value_counts().head())

Frecuencia observada en la muestra de SECOP II:
fecha_de_firma
0 days    182511
1 days      2932
2 days       127
3 days        72
4 days        25
Name: count, dtype: int64
